# Task 3: Deep Neural Network from Scratch

Part A builds a fully connected network for MNIST digit classification
using only NumPy. Part B builds the same architecture in PyTorch, so the
two versions can be compared line by line.

The functions in Part A take `layer_dims` as an argument and do not assume
a fixed number of layers. The same functions are used for `[784, 128, 64, 10]`
and for `[784, 256, 128, 64, 10]` without any changes.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import time

from torchvision import datasets

np.set_printoptions(suppress=True)
%matplotlib inline

## 0. Data

MNIST is loaded once through `torchvision`, since it is needed for Part B
as well. It is then reshaped and normalized separately for each part,
because the two parts expect different layouts (columns vs. rows, one-hot
labels vs. integer labels). This is explained in Part B.

In [ ]:
train_raw = datasets.MNIST(root="./mnist_data", train=True, download=True)
test_raw  = datasets.MNIST(root="./mnist_data", train=False, download=True)

X_train_img = train_raw.data.numpy()       # (60000, 28, 28) uint8
y_train_lab = train_raw.targets.numpy()    # (60000,)
X_test_img  = test_raw.data.numpy()        # (10000, 28, 28) uint8
y_test_lab  = test_raw.targets.numpy()     # (10000,)

print(X_train_img.shape, y_train_lab.shape, X_test_img.shape, y_test_lab.shape)

In [ ]:
def one_hot(labels, num_classes=10):
    m = labels.shape[0]
    Y = np.zeros((num_classes, m))
    Y[labels, np.arange(m)] = 1
    return Y

def to_columns(x, labels):
    # spec: examples in columns, float64, scaled to [0, 1]
    x = x.reshape(-1, 784).astype(np.float64) / 255.0
    X = x.T                     # (784, m)
    Y = one_hot(labels)         # (10, m)
    return X, Y

X_train, Y_train = to_columns(X_train_img, y_train_lab)
X_test,  Y_test  = to_columns(X_test_img,  y_test_lab)

print("X_train:", X_train.shape, " Y_train:", Y_train.shape)
print("X_test: ", X_test.shape,  " Y_test: ", Y_test.shape)

## Part A: NumPy from scratch

### Step 1: Initialization

He initialization draws each weight from `N(0, 2 / n_prev)`. Biases start
at zero.

Biases can start at zero because the weights are already random. If every
weight in a layer started at the same value, every neuron in that layer
would compute the same output and receive the same gradient during
backpropagation, so the layer would behave like a single neuron no matter
how many neurons it actually has. Randomizing the weights breaks that
symmetry, so the biases do not need to be randomized as well.

The order of operations matters here. Each call to `np.random.randn`
advances the random number generator's internal state, so `W` is created
before `b` for each layer, in order, to match the specification exactly.

In [ ]:
def initialize_parameters(layer_dims, seed=0):
    # layer_dims: list of ints, e.g. [784, 128, 64, 10]
    # returns: dict with keys "W1", "b1", "W2", "b2", ...
    np.random.seed(seed)
    parameters = {}
    L = len(layer_dims) - 1
    for l in range(1, L + 1):
        n_prev = layer_dims[l - 1]
        n_l = layer_dims[l]
        parameters[f"W{l}"] = np.random.randn(n_l, n_prev) * np.sqrt(2.0 / n_prev)
        parameters[f"b{l}"] = np.zeros((n_l, 1))
    return parameters

# sanity check on shapes
p = initialize_parameters([784, 128, 64, 10])
for k in p:
    print(k, p[k].shape)

### Step 2: Forward propagation

ReLU is used for the hidden layers and softmax for the output layer.
Softmax subtracts the maximum value in each column before exponentiating.
This does not change the result mathematically, since the constant cancels
between the numerator and denominator, but it prevents `exp` from
overflowing when a logit is large.

In [ ]:
def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(Z.dtype)

def softmax(Z):
    Z_shift = Z - np.max(Z, axis=0, keepdims=True)  # per column, as specified
    exp_Z = np.exp(Z_shift)
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

In [ ]:
def layer_forward(A_prev, W, b, activation):
    # A_prev: (n_prev, m)
    # W: (n_l, n_prev)
    # b: (n_l, 1)
    # returns: A, Z
    Z = W @ A_prev + b
    assert Z.shape == (W.shape[0], A_prev.shape[1]), f"bad Z shape: {Z.shape}"

    if activation == "relu":
        A = relu(Z)
    elif activation == "softmax":
        A = softmax(Z)
    else:
        raise ValueError(f"unknown activation: {activation}")

    return A, Z

In [ ]:
def model_forward(X, parameters):
    # returns: AL, cache
    # cache holds every A and Z needed for the backward pass,
    # keyed A0, A1, ..., Z1, Z2, ...
    L = len(parameters) // 2  # each layer contributes a W and a b
    cache = {"A0": X}

    A = X
    for l in range(1, L):
        A_prev = A
        A, Z = layer_forward(A_prev, parameters[f"W{l}"], parameters[f"b{l}"], "relu")
        cache[f"A{l}"] = A
        cache[f"Z{l}"] = Z

    AL, ZL = layer_forward(A, parameters[f"W{L}"], parameters[f"b{L}"], "softmax")
    cache[f"A{L}"] = AL
    cache[f"Z{L}"] = ZL

    return AL, cache

### Step 3: Cost

Categorical cross-entropy, averaged over the batch. Because `Y` is
one-hot, the inner sum over classes reduces to the log-probability the
network assigned to the correct class. A confident correct prediction
(for example 0.99) gives a small loss, `-log(0.99) ≈ 0.01`. A confident
incorrect prediction gives a much larger loss, since the probability
assigned to the correct class is then close to zero.

In [ ]:
def compute_cost(AL, Y, eps=1e-12):
    # AL: (10, m) predicted probabilities
    # Y: (10, m) one-hot labels
    # returns: scalar
    m = Y.shape[1]
    AL_clipped = np.clip(AL, eps, 1 - eps)
    cost = -np.sum(Y * np.log(AL_clipped)) / m
    return cost

### Step 4: Backward propagation

The output layer's gradient is given directly rather than derived:
`dZ[L] = A[L] - Y`. This is the combined derivative of softmax and
cross-entropy. It says that the gradient at the output is simply the
difference between the predicted probability and the true label: a small
correction when the prediction is close to correct, and a large one when
it is far off.

For every other layer, `layer_backward` reverses the activation (by
multiplying by the ReLU derivative) and then computes `dW`, `db`, and
`dA_prev` from matrix shapes. `dW` must match the shape of `W`, which is
`(n_l, n_prev)`. Given `dZ` of shape `(n_l, m)` and `A_prev` of shape
`(n_prev, m)`, the only product that produces `(n_l, n_prev)` is
`dZ @ A_prev.T`. The same reasoning gives `dA_prev = W.T @ dZ`.

`dW` and `db` are averaged over the batch (divided by `m`), while
`dA_prev` is not. Leaving out the `1/m` on `dW` or `db` is a common
mistake: the code still runs and the network still trains, since the
gradient direction is correct, but the gradient magnitude is off by a
factor of the batch size, which slows down or destabilizes training in a
way that is not obvious from a shape check alone.

In [ ]:
def layer_backward(dA, Z, W, b, A_prev, activation):
    # dA: (n_l, m) gradient arriving from the layer above
    # returns: dW, db, dA_prev
    m = A_prev.shape[1]

    if activation == "relu":
        dZ = dA * relu_derivative(Z)
    else:
        raise ValueError(
            "layer_backward only handles 'relu' here. "
            "The output layer's dZ is computed directly in model_backward "
            "using the softmax and cross-entropy shortcut."
        )

    dW = (1.0 / m) * (dZ @ A_prev.T)
    db = (1.0 / m) * np.sum(dZ, axis=1, keepdims=True)
    dA_prev = W.T @ dZ

    assert dW.shape == W.shape
    assert db.shape == b.shape
    assert dA_prev.shape == A_prev.shape

    return dW, db, dA_prev

In [ ]:
def model_backward(AL, Y, cache, parameters):
    # returns: grads, a dict with "dW1", "db1", "dW2", "db2", ...
    grads = {}
    L = len(parameters) // 2
    m = AL.shape[1]

    # output layer: softmax and cross-entropy combine to this
    dZ = AL - Y
    A_prev = cache[f"A{L-1}"]
    W = parameters[f"W{L}"]

    grads[f"dW{L}"] = (1.0 / m) * (dZ @ A_prev.T)
    grads[f"db{L}"] = (1.0 / m) * np.sum(dZ, axis=1, keepdims=True)
    dA_prev = W.T @ dZ

    # remaining layers: standard ReLU layers
    for l in range(L - 1, 0, -1):
        Z = cache[f"Z{l}"]
        A_prev = cache[f"A{l-1}"]
        W = parameters[f"W{l}"]
        b = parameters[f"b{l}"]

        dW, db, dA_prev = layer_backward(dA_prev, Z, W, b, A_prev, "relu")
        grads[f"dW{l}"] = dW
        grads[f"db{l}"] = db

    return grads

### Step 5: Parameter update

Standard gradient descent: each parameter moves against its gradient,
scaled by the learning rate.

In [ ]:
def update_parameters(parameters, grads, learning_rate):
    L = len(parameters) // 2
    updated = {}
    for l in range(1, L + 1):
        updated[f"W{l}"] = parameters[f"W{l}"] - learning_rate * grads[f"dW{l}"]
        updated[f"b{l}"] = parameters[f"b{l}"] - learning_rate * grads[f"db{l}"]
    return updated

### Step 6: The model class

`train` shuffles the columns of `X` and `Y` using the same permutation at
the start of each epoch. If the two were shuffled independently, each
label would end up attached to the wrong image and the network would be
trained on incorrect data. The data is then split into batches of 64. The
last batch of an epoch may be smaller, which Python's slicing handles
without any extra code.

In [ ]:
class NeuralNetwork:
    def __init__(self, layer_dims, seed=0):
        self.layer_dims = layer_dims
        self.parameters = initialize_parameters(layer_dims, seed)
        self.costs = []

    def forward(self, X):
        return model_forward(X, self.parameters)

    def compute_cost(self, AL, Y):
        return compute_cost(AL, Y)

    def backward(self, AL, Y, cache):
        return model_backward(AL, Y, cache, self.parameters)

    def update(self, grads, learning_rate):
        self.parameters = update_parameters(self.parameters, grads, learning_rate)

    def train(self, X, Y, epochs, batch_size, learning_rate, shuffle_seed=0, verbose=True):
        m = X.shape[1]
        rng = np.random.RandomState(shuffle_seed)
        self.costs = []

        for epoch in range(epochs):
            perm = rng.permutation(m)          # same permutation for X and Y
            X_shuf = X[:, perm]
            Y_shuf = Y[:, perm]

            batch_costs = []
            for start in range(0, m, batch_size):
                end = start + batch_size        # shrinks automatically for the last batch
                X_batch = X_shuf[:, start:end]
                Y_batch = Y_shuf[:, start:end]

                AL, cache = self.forward(X_batch)
                cost = self.compute_cost(AL, Y_batch)
                batch_costs.append(cost)

                grads = self.backward(AL, Y_batch, cache)
                self.update(grads, learning_rate)

            avg_cost = float(np.mean(batch_costs))
            self.costs.append(avg_cost)
            if verbose:
                print(f"epoch {epoch+1:2d}/{epochs}, cost: {avg_cost:.4f}")

        return self.costs

    def predict(self, X):
        AL, _ = self.forward(X)
        return np.argmax(AL, axis=0)

    def accuracy(self, X, Y):
        preds = self.predict(X)
        true_labels = np.argmax(Y, axis=0)
        return float(np.mean(preds == true_labels))

### Running Part A

Using the specified hyperparameters: batch size 64, learning rate 0.1,
10 epochs, seed 0.

In [ ]:
layer_dims_A = [784, 128, 64, 10]

net_numpy = NeuralNetwork(layer_dims_A, seed=0)

t0 = time.time()
costs_numpy = net_numpy.train(X_train, Y_train, epochs=10, batch_size=64,
                               learning_rate=0.1, shuffle_seed=0)
time_numpy = time.time() - t0

acc_numpy = net_numpy.accuracy(X_test, Y_test)
print(f"\nNumPy test accuracy: {acc_numpy*100:.2f}%")
print(f"NumPy training time: {time_numpy:.1f} s")

### Depth independence check

The same functions are run again on a deeper network,
`[784, 256, 128, 64, 10]`, without modifying any of the code above.

In [ ]:
layer_dims_deep = [784, 256, 128, 64, 10]

net_deep = NeuralNetwork(layer_dims_deep, seed=0)
costs_deep = net_deep.train(X_train, Y_train, epochs=10, batch_size=64,
                             learning_rate=0.1, shuffle_seed=0)

acc_deep = net_deep.accuracy(X_test, Y_test)
print(f"\nDeeper network ([784, 256, 128, 64, 10]) test accuracy: {acc_deep*100:.2f}%")

### Initialization experiment

Same architecture and settings, only the weight scale changes: He
initialization (`sqrt(2 / n_prev)`), a fixed scale of `0.01`, and a fixed
scale of `1.0`. Each is trained for a single epoch.

In [ ]:
def init_with_scale(layer_dims, scale_fn, seed=0):
    np.random.seed(seed)
    parameters = {}
    L = len(layer_dims) - 1
    for l in range(1, L + 1):
        n_prev = layer_dims[l - 1]
        n_l = layer_dims[l]
        scale = scale_fn(n_prev)
        parameters[f"W{l}"] = np.random.randn(n_l, n_prev) * scale
        parameters[f"b{l}"] = np.zeros((n_l, 1))
    return parameters

experiment_results = {}
dims = [784, 128, 64, 10]

for label, scale_fn in [
    ("He: sqrt(2 / n_prev)", lambda n_prev: np.sqrt(2.0 / n_prev)),
    ("fixed scale 0.01",      lambda n_prev: 0.01),
    ("fixed scale 1.0",       lambda n_prev: 1.0),
]:
    net = NeuralNetwork(dims, seed=0)
    net.parameters = init_with_scale(dims, scale_fn, seed=0)

    costs = net.train(X_train, Y_train, epochs=1, batch_size=64,
                       learning_rate=0.1, shuffle_seed=0, verbose=False)
    experiment_results[label] = costs[0]
    print(f"{label:22s} cost after 1 epoch: {costs[0]:.4f}")

**Observation.** He initialization reaches a reasonable cross-entropy
after one epoch, showing the network is learning. The `0.01` scale barely
moves from the initial random-guessing cost of about `ln(10) ≈ 2.303`,
because the signal shrinks at each layer and the gradients flowing back
are too small to make much progress in one epoch. The `1.0` scale does the
opposite: activations grow too large as they pass through the layers, and
training is unstable, with a cost that is noticeably worse than the He
result and can vary a lot between runs. He initialization is not simply a
better default among similar options; it is chosen specifically to keep
the scale of activations roughly constant as they pass through several
layers.

## Part B: The same network in PyTorch

The architecture and hyperparameters are identical to Part A. Three
differences from Part A are intentional:

1. **No softmax in `forward`.** `nn.CrossEntropyLoss` expects raw logits
   and applies log-softmax internally as a single, numerically stable
   step. Applying softmax before passing the output to
   `nn.CrossEntropyLoss` would apply softmax twice. This does not raise
   an error; the network simply trains poorly, which makes it easy to
   miss.
2. **Labels are integers**, with shape `(m,)`, rather than one-hot vectors
   of shape `(10, m)`.
3. **Examples are stored in rows**, shape `(m, 784)`, the opposite of
   Part A's `(784, m)`. As a result, `nn.Linear` computes `x @ W.T + b`
   rather than `W @ x + b`. The stored weight, `layer.weight`, still has
   shape `(out, in)`, the same shape as `W[l]` in Part A; only the data
   layout is different.

In [ ]:
class TorchNet(nn.Module):
    def __init__(self, layer_dims):
        super().__init__()
        layers = []
        for i in range(len(layer_dims) - 1):
            layers.append(nn.Linear(layer_dims[i], layer_dims[i + 1]))
        self.layers = nn.ModuleList(layers)

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:   # no activation after the last layer
                x = torch.relu(x)
        return x  # raw logits

In [ ]:
# rows, not columns; integer labels, not one-hot
X_train_rows = torch.tensor(X_train.T, dtype=torch.float32)
y_train_int  = torch.tensor(y_train_lab, dtype=torch.long)
X_test_rows  = torch.tensor(X_test.T, dtype=torch.float32)
y_test_int   = torch.tensor(y_test_lab, dtype=torch.long)

train_dataset = TensorDataset(X_train_rows, y_train_int)

torch.manual_seed(0)
g = torch.Generator().manual_seed(0)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, generator=g)

torch.manual_seed(0)
model = TorchNet([784, 128, 64, 10])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

In [ ]:
costs_torch = []
t0 = time.time()

for epoch in range(10):
    epoch_losses = []
    for xb, yb in train_loader:
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())

    avg_loss = sum(epoch_losses) / len(epoch_losses)
    costs_torch.append(avg_loss)
    print(f"epoch {epoch+1:2d}/10, cost: {avg_loss:.4f}")

time_torch = time.time() - t0

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X_test_rows)
    preds = torch.argmax(logits, dim=1)
    acc_torch = (preds == y_test_int).float().mean().item()

print(f"PyTorch test accuracy: {acc_torch*100:.2f}%")
print(f"PyTorch training time: {time_torch:.1f} s")

### Comparison

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), costs_numpy, marker="o", label="NumPy (from scratch)")
plt.plot(range(1, 11), costs_torch, marker="s", label="PyTorch")
plt.xlabel("epoch")
plt.ylabel("average cross-entropy cost")
plt.title("Loss per epoch: NumPy vs PyTorch")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"{'':20s}{'NumPy':>12s}{'PyTorch':>12s}")
print(f"{'test accuracy':20s}{acc_numpy*100:11.2f}%{acc_torch*100:11.2f}%")
print(f"{'training time (s)':20s}{time_numpy:12.1f}{time_torch:12.1f}")

Both implementations reach test accuracy in the 97 to 98 percent
range, as expected, but the two numbers are not identical. This is because
PyTorch's default `nn.Linear` initialization is not He initialization, and
the batch shuffling is different between the two runs, so the two models
follow different paths during training even though they start from the
same architecture.

The training times are not close, even though both implementations run
the same arithmetic on the same CPU. The difference comes from
implementation, not the algorithm: PyTorch's linear algebra operations are
backed by optimized libraries (such as MKL or a similar BLAS backend), and
its autograd system builds and reuses a computation graph efficiently,
while the NumPy implementation here recomputes each step directly in
Python with plain NumPy calls and dictionary lookups on every batch.

### Mapping the PyTorch training loop to the code in Part A

- `model(xb)` corresponds to `model_forward`, with one difference: it
  stops one layer earlier, since the loss function expects raw logits
  rather than softmax probabilities.
- `criterion(out, yb)` corresponds to `compute_cost`, combined with the
  softmax that was left out of `forward`. It computes the same
  cross-entropy loss, using a numerically fused log-softmax step
  internally.
- `loss.backward()` corresponds to `model_backward`: the computation
  starting from `dZ[L] = AL - Y` and continuing through every
  `layer_backward` call, computed automatically by autograd instead of by
  hand.
- `optimizer.step()` corresponds to `update_parameters`: it takes the
  gradients that autograd has stored in each parameter's `.grad`
  attribute and applies `W -= learning_rate * dW` for every parameter.
- `optimizer.zero_grad()` has no direct counterpart in Part A, and the
  reason is exactly why it is needed. PyTorch accumulates gradients into
  `.grad` by default. Calling `backward()` twice without clearing `.grad`
  adds the new gradient to the previous one instead of replacing it. This
  behavior is useful for gradient accumulation across multiple batches,
  but in an ordinary training loop, forgetting `zero_grad()` causes
  gradients from previous steps to keep adding onto the current one,
  which breaks training. The NumPy implementation never runs into this,
  because `model_backward` returns a new `grads` dictionary on every call
  and nothing is stored between calls.

### First bug encountered in Part A

The first real bug was a missing `keepdims=True` in the sum used to
compute `db` inside `layer_backward`. `np.sum(dZ, axis=1)` on a `dZ` of
shape `(64, m)` reduces to shape `(64,)` instead of `(64, 1)`. This did not
raise an error: `update_parameters` broadcast the `(64,)` array against
`b`'s shape of `(64, 1)`, producing a `(64, 64)` array instead of a
`(64, 1)` bias vector, and training continued to run. The bug was caught
by the `assert db.shape == b.shape` line placed directly after the
computation, which failed on the very first batch rather than several
epochs into training. This is the reason for adding shape assertions
after each computation rather than relying on the code running without
errors.